# V28: unchanged native batches with CPU prefetch

One bounded CPU producer prepares the next batch while the consumer runs the
original model. Packing remains8192 and individual conversation cap4096.
CPU proofs measure batch identity, not GPU equivalence, speed or score.
[Plan](../docs/SPAN_028_PLAN.md) · [CPU proof](../docs/SPAN_028_CPU_PROOF.md) ·
[Final result](../docs/SPAN_028_RESULTS.md) ·
[Preserved packing failure](../docs/SPAN_027_RESULTS.md).

In [1]:
from pathlib import Path
import json
import pandas as pd
REPO=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
PUB=REPO/'results/span_028';proof=json.loads((PUB/'cpu_audit.json').read_text())
rows=[]
for role,cohorts in proof['actual_native_cohorts'].items():
 for name,v in cohorts.items():
  rows.append({'role':role,'cohort':name,'rows':v['rows'],'microbatches':v['micro_batches'],
   'host_batches_exact':v['exact_host_batches'],'row_order_exact':v['exact_row_order'],
   'positions_exact':v['exact_positions'],'local_sync_preparation_s':v['local_synchronous_preparation_seconds']})
display(pd.DataFrame(rows))
print('CPU proof elapsed:',proof['elapsed_cpu_seconds'],'seconds; cap:',proof['cpu_cap_seconds'])
print('Packing:',proof['packing_tokens'],'; producer threads:',proof['producer_threads'],'; queue:',proof['queue_depth'])
print('No labels, GPU inference, probability identity, speedup or quality measured.')
print('Preparation times are laptop measurements during an audit, not expected Kaggle savings.')
for name in ['prefetch_tests.json','profile_tests.json','forward_loop_rehearsal.json','prefetch_profile_cell_rehearsals.json']:
 path=PUB/name
 if path.exists():print(name,json.loads(path.read_text()))
decision=PUB/'profile_audit.json'
if decision.exists():
 result=json.loads(decision.read_text());print('Engineering decision:',result['decision'])
 print('All raw cohorts max probability difference:',max(result.get('canonical_max_abs',{'unknown':float('nan')}).values()))
 print('Qualified:',result['engineering_gate_passed'],'; official eligible:',result['official_submission_eligible'])
 if 'runtime' in result:display(pd.DataFrame([result['runtime']]))
 if 'peak_memory_fractions' in result:display(pd.DataFrame([result['peak_memory_fractions']]))
else:print('GPU inference and its terminal engineering audit are pending. Preparation evidence above does not admit H73.')
description=PUB/'runtime_description.json'
if description.exists():
 d=json.loads(description.read_text());display(pd.DataFrame(d['measured_uniform_timings']).T)
 print('Separate two-role feasibility screen (not qualification):',d['two_role_feasibility'])
inventory=PUB/'terminal_inventory.json'
if inventory.exists():print('Terminal inventory/quota:',{k:v for k,v in json.loads(inventory.read_text()).items() if k in ['observed_at','owned_notebook_count','quota','nonterminal_jobs']})
board=PUB/'final_leaderboard.json'
if board.exists():print('Complete closing leaderboard:',json.loads(board.read_text()))
status=PUB/'jobs/prefetch_profile/status.json'
if status.exists():print('Last observed exact GPU job:',json.loads(status.read_text()))
print('H73 three-role60/40prior remains deferred. If engineering failed, no replay or official is admitted; the distinct two-role proposal requires its own future contract and qualification.')

,role,cohort,rows,microbatches,host_batches_exact,row_order_exact,positions_exact,local_sync_preparation_s
0,m0,warmup,80,9,True,True,True,0.447334
1,m0,control,256,33,True,True,True,1.088361
2,m0,uniform,512,67,True,True,True,2.468255
3,m0,long,64,63,True,True,True,2.043294
4,m0,test,3,1,True,True,True,0.022970
5,m3,warmup,80,9,True,True,True,0.304466
6,m3,control,256,33,True,True,True,1.082574
7,m3,uniform,512,65,True,True,True,2.290881
8,m3,long,64,63,True,True,True,1.953235
9,m3,test,3,1,True,True,True,0.029615


CPU proof elapsed: 31.847062963061035 seconds; cap: 300
Packing: 8192 ; producer threads: 1 ; queue: 2
No labels, GPU inference, probability identity, speedup or quality measured.
Preparation times are laptop measurements during an audit, not expected Kaggle savings.
prefetch_tests.json {'passed': True, 'tests': 7, 'runner': 'PYTHONPATH=src .venv/bin/python -m unittest discover -s tests -p test_cpu_prefetch.py -v', 'observed_test_seconds': 0.104, 'source_sha256': {'src/arena/cpu_prefetch.py': 'da7502af819a86c421717f68d40c4a9892cad60bc2f49279247805697be30c35', 'tests/test_cpu_prefetch.py': '54fab79e57894f8b0761518ab65e48335c7622cf8e9b55e2fc630774b6cbdcc5'}}
profile_tests.json {'passed': True, 'tests': 6, 'runner': 'PYTHONPATH=src .venv/bin/python -m unittest discover -s tests -p test_span028_profile.py -v', 'observed_test_seconds': 0.133, 'source_sha256': {'src/arena/span028_checks.py': '3a685a55eb37d053caa7305ab8a56f07117b43e345b69933fef77fb70e680c48', 'src/arena/span028_profile.py': '

,projection_seconds,steady_projection_seconds,new_execution_uniform_seconds,cold_seconds,full_warmup_seconds,other_seconds,measured_role_seconds,session_seconds,margin,reserve_seconds,packing_tokens,synchronous_throughput_is_comparator_not_new_mode_floor,divides_already_parallel_wall_time_by_gpu_count,runtime_gate_passed
0,32973.468183,31751.281308,"{'m0': 293.90902281700005, 'm3': 213.175659101...",407.983022,158.354866,55.848986,1682.940585,1726.087086,1.2,600,8192,True,False,False


,m0,m3,m0_reverse
0,0.701233,0.568409,0.699275


,rows,cohort_wall_seconds,preparation_seconds,producer_queue_wait_seconds,consumer_queue_wait_seconds,forward_launch_transfer_hostcopy_wall_seconds
m0,512.0,293.909023,4.498641,227.286700,0.600896,293.254233
m3,512.0,213.175659,4.587406,162.386153,0.663570,212.453083
m0_reverse,83.0,34.803852,0.420332,0.000072,0.390910,34.406052


Separate two-role feasibility screen (not qualification): {'roles': ['m0', 'm3'], 'steady_seconds': 29711.99308113282, 'cold_seconds': 293.4325864829997, 'full_warmup_seconds': 105.48687213099964, 'other_seconds': 55.848985514120784, 'margin': 1.2, 'reserve_seconds': 600, 'projection_seconds': 30766.76152526094, 'candidate_qualified': False, 'quality_measured': False, 'official_submission_eligible': False, 'interpretation': 'Uses both actually measured base roles and retained startup floors; no reverse pass or hypothetical faster timing. A new two-role candidate still needs actual qualification.'}
Terminal inventory/quota: {'observed_at': '2026-11-08T21:59:43.868136+00:00', 'quota': {'used_seconds': 31657.474, 'reserved_seconds': 0.0, 'allowed_seconds': 108000.0, 'refresh_time': '2026-11-14 00:00:00', 'paid_scaling': False, 'remaining_seconds': 76342.526}, 'owned_notebook_count': 73, 'nonterminal_jobs': []}
Complete closing leaderboard: {'observed_at': '2026-11-08T21:57:35.157387+00:00